# 第11回 演習：教師なし学習とPCA（固有顔）（解答例・教員用）

## 今日の分析目標

**ラベルなしで、4096次元の顔を少数の軸で捉え、圧縮したい。**

この演習では、PCA で顔画像を固有顔に分解し、累積寄与率で次元を減らし、少数の成分で顔を再構成する流れを、自分の手で動かします。主成分がなぜ「分散が最大の方向」になるのか、その正体である固有値分解まで踏み込みます。TODOに取り組みながら、最後の「目標に答えられたか」で振り返りましょう。


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_olivetti_faces
from sklearn.decomposition import PCA

plt.rcParams['font.size'] = 11
faces = fetch_olivetti_faces(shuffle=False)
X = faces.data          # 400枚 × 4096ピクセル
y = faces.target        # 誰の顔か（0〜39）。今回は原則使わない
pca = PCA().fit(X)
print(f'{X.shape[0]}枚, {X.shape[1]}ピクセル, {len(np.unique(y))}人')

## 1. 固有顔を見る

主成分を画像に戻すと「固有顔」になります。

### 深掘り：固有顔の正体——共分散行列の固有値分解 $\Sigma v = \lambda v$

上のセルで並んだ「固有顔」は、なぜ**顔の形**をしているのでしょう。名前に「固有」とつくのは飾りではなく、PCA の主成分が**共分散行列の固有ベクトル**そのものだからです。ここを一段だけ数式で押さえておくと、以降の再構成も圧縮も一本の線でつながります。

**まず土俵をそろえる（中心化）**　400枚の顔を平均した**平均顔** $\bar{x}$（このセルの左上の画像）を各画像から引き、$\tilde{x} = x - \bar{x}$ という「平均からのズレ」に直します。以降はこのズレだけを扱います。画像は全ピクセルが 0〜1 の同じ明るさスケールなので、平均を引く**中心化**はしても、ピクセルごとに割り算する標準化はしません（この判断の理由は次節で詳しく述べます）。

**共分散行列 $\Sigma$ とは**　4096個のピクセルについて、「ピクセル $j$ が平均より明るいとき、ピクセル $k$ も一緒に明るくなりやすいか」を全ペアで表した $4096\times4096$ の表が、共分散行列 $\Sigma$ です。対角がそのピクセル単体の分散、非対角がピクセル間の連動を表します。顔では「額が明るいとき頬も明るい」のように、多くのピクセルが連動して動きます。この連動の構造こそ、PCA が取り出したいものです。

**主成分＝分散が最大の方向、その骨子**　4096次元の空間で、ある単位ベクトル $v$（長さ1の向き）を1本引きます。各顔のズレ $\tilde{x}$ をこの向きに射影した値 $\tilde{x}\cdot v$ の**分散**は、$v^{\top}\Sigma v$ と書けます。PCA は「射影したとき最も散らばる（分散が最大の）向き」を探すので、

$$
\max_{v}\ v^{\top}\Sigma v \quad \text{s.t.}\quad \lVert v \rVert = 1
$$

という問題を解きます。長さを1に縛ったうえで最大化するので、ラグランジュの未定乗数 $\lambda$ を使うと、最大値を与える $v$ は

$$
\Sigma v = \lambda v
$$

を満たします——これが**固有値方程式**で、$v$ が $\Sigma$ の**固有ベクトル**、$\lambda$ がその**固有値**です。つまり「分散が最大の方向」を探す作業は、共分散行列の固有ベクトルを求める作業と**同じもの**でした。さらに、この $v$ に沿った分散を計算し直すと $v^{\top}\Sigma v = v^{\top}(\lambda v) = \lambda$。**固有値 $\lambda$ が、その方向に沿った分散の量そのもの**なのです。だから固有値のいちばん大きい向きが第1主成分、次が第2主成分……と並びます。

**第2・第3主成分はどこから来るか**　第2主成分は「第1主成分と直交する向きのうち、次に分散が大きい向き」でした。これも同じ $\Sigma v = \lambda v$ の、2番目に大きい固有値に対応する固有ベクトルとして自動的に出てきます。対称行列 $\Sigma$ の固有ベクトルは互いに直交するので、主成分が直交する（重複しない情報を持つ）のは、固有ベクトルの性質のあらわれです。厳密な最大化の証明（スペクトル分解の定理や、逐次的に直交制約下で最大化すると固有値の大きい順に主成分が並ぶこと）は、**詳しくは lesson_MVA『主成分分析』回**に譲ります。ここでは「分散最大＝固有値最大」「直交＝対称行列の固有ベクトル」という骨格をつかめれば十分です。

**この実データの数字で確かめる**　第1主成分の固有値は $\lambda_1 \approx 18.84$。これは下のセルで確かめるとおり、第1主成分に射影したスコアの分散と**ぴたり一致**します（固有値＝その方向の分散、の実証）。固有値をすべて足すと $\sum_j \lambda_j \approx 79.12$ で、これは元の4096ピクセルの分散の総和と一致します。固有値分解は**分散の総量を保ったまま、それを少数の向きへ組み替えているだけ**なのです。だから各主成分の**寄与率**は $\lambda_1 / \sum_j \lambda_j = 18.84/79.12 \approx 23.8\%$ と計算でき、上のコードが表示した「第1主成分で約24%」の中身がこれです。第2主成分は約14%、第3主成分は約8%と、後ろほど固有値が小さく、寄与率も小さくなっていきます。

**固有ベクトルを画像に戻すと固有顔**　固有ベクトル $v$ は 4096個の数字の並び。それを $64\times64$ に並べ直したものが**固有顔**です。つまり固有顔は、抽象的な主成分を**そのまま絵にしたもの**にほかなりません。そして任意の顔は

$$
x \;=\; \bar{x} \;+\; \sum_k s_k\, v_k, \qquad s_k = \tilde{x}\cdot v_k
$$

と、**平均顔に固有顔を重み付きで足し合わせた形**で表せます。重み $s_k$（スコア）は「その顔がその固有顔をどれだけ含むか」。たとえば1枚目の顔 `X[0]` のスコアは、第1〜5成分でおよそ $(6.43,\ 0.70,\ -1.43,\ 1.28,\ -2.56)$。これは「この顔 ＝ 平均顔 ＋ 6.43×固有顔1 ＋ 0.70×固有顔2 ＋ …」という**レシピ**を読んでいるのと同じです。第1成分の重みが飛び抜けて大きいのは、固有顔1が明るさや大きな陰影という「どの顔にも強く効く成分」だから。この足し算を上位 $K$ 個で打ち切れば、4096個の数字を $K$ 個のスコアへ圧縮したことになります。次の節では、この打ち切りを実際に目で確かめます。

**別の見方：最良の低ランク近似（SVD）**　同じことは、中心化したデータ行列そのものの**特異値分解（SVD）**としても見えます。上位 $K$ 個の主成分で作った再構成は、元データに**いちばん近い**（二乗誤差が最小の）$K$本の軸による近似になっている——これは行列近似の基本定理（エッカート＝ヤング）が保証します。「分散を最大に残す向きを選ぶ」ことと「元画像との誤差を最小にする向きを選ぶ」ことが**表裏一体**だという事実が、次節の再構成が一番よく効く理由です。

**つまずきどころ：固有顔を1枚だけで意味づけすぎない**　固有顔1は明るさ、固有顔2は左右の向き……と読みたくなりますが、各固有顔は多数のピクセルの混ぜ合わせで、前方の成分ほど大域的（明るさ・向き）、後方ほど細部（しわ・メガネの縁）を担う傾向がある、という程度に留めるのが安全です。「この1枚が“目”を表す」といった過度な意味づけは、次節・次々節のつまずき（標準化と符号）と並ぶ、PCA 解釈の落とし穴です。

**計算上の補足：$4096\times4096$ の行列は作らない**　共分散行列 $\Sigma$ は $4096\times4096$ と巨大ですが、幸い画像はたった400枚。`sklearn` の `PCA` は $\Sigma$ を明示的に組み立てず、中心化したデータ行列の**特異値分解（SVD）**で同じ固有ベクトル・固有値を効率よく求めます。だから4096次元でも一瞬で終わります。そして忘れてはならないのは、ここまで**ラベル `y`（誰の顔か）を一度も使っていない**こと。分散という手がかりだけで軸を見つける——これが教師なし学習らしさです（`y` を使うのは、次節でわざわざ人物当てを試すときだけ）。


In [ ]:
# 深掘りの数値確認：固有値 λ = その主成分に沿った分散、そして分散の総量は保存される
scores = (X - X.mean(0)) @ pca.components_.T   # 各顔を各主成分へ射影したスコア
print(f'第1主成分の固有値 λ1            : {pca.explained_variance_[0]:.3f}')
print(f'第1主成分スコアの分散(不偏)     : {scores[:, 0].var(ddof=1):.3f}  ← λ1 と一致')
print(f'固有値の総和 Σλ                 : {pca.explained_variance_.sum():.2f}')
print(f'元の4096ピクセルの分散の総和     : {X.var(0, ddof=1).sum():.2f}  ← Σλ と一致（総量は保存）')
print(f'第1主成分の寄与率 λ1/Σλ         : {pca.explained_variance_ratio_[0]:.3f}')


In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(10, 4.2))
axes[0, 0].imshow(X.mean(0).reshape(64, 64), cmap='gray')
axes[0, 0].set_title('平均顔'); axes[0, 0].axis('off')
for i, ax in enumerate(axes.ravel()[1:]):
    ax.imshow(pca.components_[i].reshape(64, 64), cmap='gray')
    ax.set_title(f'固有顔{i+1}'); ax.axis('off')
plt.tight_layout(); plt.show()

### TODO①：累積寄与率で「50個で何割か」を確かめる

`pca.explained_variance_ratio_` を累積して、上位50個の主成分で全体の何割の情報を保てるかを計算・表示してください。

In [ ]:
# TODO: 累積寄与率を計算し、上位50個で保てる情報の割合を表示してください

# 解答例①：累積寄与率
cum = np.cumsum(pca.explained_variance_ratio_)
print(f'上位50個で保てる情報: {cum[49]:.1%}')
for k in [5, 10, 20, 50, 100]:
    print(f'  上位{k:3d}個: {cum[k-1]:.1%}')


## 2. 顔を再構成する（圧縮の実感）

平均顔に固有顔を少しずつ足して、1枚の顔を組み立て直します。

### 深掘り：再構成の式と、累積寄与率が測る「失う情報」

**再構成の式を1行ずつ**　上のセルの `rec = X.mean(0) + (face @ pca.components_[:k].T) @ pca.components_[:k]` は、前節の $x = \bar{x} + \sum_k s_k v_k$ を上位 $k$ 個で打ち切ったものです。`face = X[0] - X.mean(0)` が中心化した $\tilde{x}$、`face @ pca.components_[:k].T` が各固有顔への射影＝スコアの並び $(s_1,\dots,s_k)$、それに `@ pca.components_[:k]` を掛けると $\sum_{i\le k} s_i v_i$ になり、最後に平均顔 $\bar{x}$ を足し戻す。**「平均顔＋固有顔の重み付き和」を、コードがそのまま実行している**わけです。

**捨てた分＝失う情報の正体**　上位 $K$ 個で打ち切ると、残りの $\sum_{k>K} s_k v_k$ を捨てます。固有顔どうしは直交（互いに直角で無相関）しているため、捨てた分の大きさは主成分ごとにきれいに分解でき、全画像で平均した**再構成誤差の割合は $1-$（累積寄与率）に等しく**なります。式で書けば、捨てる情報の割合 $= \sum_{k>K}\lambda_k / \sum_j \lambda_j$。前節で見た「PCA は元画像との誤差を最小にする向きから選ぶ」ことの帰結で、TODO① で計算する累積寄与率と、この節で見た再構成の粗さは、**同じ一つの数字を別の窓から見たもの**なのです。

**実データの対応**　成分数 $K$ ごとに、その成分単体の寄与率・累積寄与率・失う情報を並べると次のようになります（`pca.explained_variance_ratio_` の実行値）。

| 成分数 $K$ | その成分の寄与率 | 累積寄与率（保てる） | 失う情報 | 再構成のようす |
|---:|---:|---:|---:|---|
| 1 | 23.8% | 23.8% | 76.2% | ほぼ平均顔 |
| 5 | 3.6% | 54.4% | 45.6% | ぼんやり、誰か曖昧 |
| 20 | 0.7% | 76.3% | 23.7% | 目鼻立ちが出る |
| 50 | 0.2% | 87.4% | 12.6% | 個人が分かる |
| 100 | 0.1% | 93.5% | 6.5% | ほぼ元画像 |

$K=50$ で累積 **87.4%**、失う情報は約 **12.6%**。だから50成分の再構成は、細部は少しぼやけても顔立ちはしっかり残ります。$K=100$ では **93.5%**（損失 6.5%）まで上がり、元画像とほとんど見分けがつきません。上の画像で「成分を増やすほど元に近づく」のは、この表の累積寄与率が1へ近づき、失う情報が減っていく様子そのものです。

**発展：累積寄与率を「次元を決めるつまみ」に使う**　何成分まで残すかは、累積寄与率を目標ラインに合わせて決められます。この顔データでは、**80%なら27成分、90%なら66成分、95%なら123成分、99%なら260成分**が必要です。表からも分かるとおり、第1成分だけで24%、上位5個で54%と前半の主成分がぐんぐん稼ぎ、20個を過ぎると1個あたりの寄与率は1%を切ってなだらかになります。この「最初は急でやがて平ら」の形がスクリープロット（がれ場の図）で、寄与率がガクッと落ちて平坦になる肘のあたりを目安にしたり、用途に必要な精度から逆算したりして次元を選びます。圧縮率と情報保持のトレードオフを、この一本の曲線で操作できるのが便利なところです。


**圧縮を数で数える**　1枚の顔はもともと4096個の数字。50成分に落とすと、1枚あたり保持するのは**50個のスコア**だけです（平均顔と50枚の固有顔は全400枚で共有する“辞書”なので、枚数が増えるほど1枚あたりの取り分は50個へ近づく）。4096が50へ、約80分の1。この身軽さのまま87%の情報が残り、顔立ちも保てるからこそ、次節の「50次元でも人物を当てられる」が効いてきます。
**つまずきどころ：画像では標準化しない**　正則化の回では「罰金の前に必ず標準化」でしたが、PCA では**中心化はするが、画像ではふつう標準化しない**——ここは混同しやすい要所です。標準化はピクセルごとにその標準偏差で割る操作ですが、顔画像の四隅や背景のピクセルは**ほとんど動かない（標準偏差がごく小さい）**。小さい値で割ると、そうした**背景のわずかなちらつきが過大に増幅**され、PCA が意味のない背景ノイズを主成分に拾ってしまいます。全ピクセルが 0〜1 の同じ単位に乗っているこのデータでは、平均顔を引く中心化だけで土俵はそろっており、標準化は害になります（`PCA` は中心化は自動で行い、標準化はしません）。逆に、単位のバラバラな**表データ**では標準化が要る——「標準化するか否かはデータの素性しだい」で、画像だから省ける、と理解するのが正解です。


In [ ]:
face = X[0] - X.mean(0)
fig, axes = plt.subplots(1, 5, figsize=(11, 2.6))
axes[0].imshow(X[0].reshape(64, 64), cmap='gray'); axes[0].set_title('元画像'); axes[0].axis('off')
for ax, k in zip(axes[1:], [5, 20, 50, 100]):
    rec = X.mean(0) + (face @ pca.components_[:k].T) @ pca.components_[:k]
    ax.imshow(rec.reshape(64, 64), cmap='gray'); ax.set_title(f'{k}成分'); ax.axis('off')
plt.tight_layout(); plt.show()

### TODO②：好きな枚数・好きな成分数で再構成する

`X` の別の顔（例：`X[100]`）を選び、成分数を変えて（例：10成分と200成分）再構成し、元画像と並べて表示してください。成分数を増やすと、どう変わりますか？

In [ ]:
# TODO: 好きな顔を選び、成分数を変えて再構成し、元画像と並べて表示してください

# 解答例②：別の顔を10成分と200成分で再構成
face2 = X[100] - X.mean(0)
fig, axes = plt.subplots(1, 3, figsize=(7, 2.6))
axes[0].imshow(X[100].reshape(64, 64), cmap='gray'); axes[0].set_title('元画像'); axes[0].axis('off')
for ax, k in zip(axes[1:], [10, 200]):
    rec = X.mean(0) + (face2 @ pca.components_[:k].T) @ pca.components_[:k]
    ax.imshow(rec.reshape(64, 64), cmap='gray'); ax.set_title(f'{k}成分'); ax.axis('off')
plt.tight_layout(); plt.show()
# → 成分が多いほど元画像に近づく


## 3. 次元を減らして、人物を当てる

4096次元と、PCAで減らした50次元で、人物当ての正解率を比べます（ここだけラベル y を使います）。

### 深掘り：50次元でも人物を見分けられる理由／符号の任意性と白色化

**なぜ80分の1に減らしても正解率が落ちないのか**　人物を見分ける手がかり（目鼻立ちや輪郭の個人差）は、まさに**顔どうしが大きく散らばる方向**に乗っています。PCA は分散の大きい向きから順に主成分を並べるので、上位50成分は「人によく効く違い」をほぼ取り切っています。捨てた約12.6%の裾は、細かなテクスチャやノイズが主で、識別にはあまり効きません。だから 4096次元でも50次元でも正解率は **0.972 のまま**。PCA が「情報の多い順」に軸を並べてくれる性質が、そのまま**識別に効く情報を優先して残す**ことにつながっているのです。しかも次元が80分の1になれば、ロジスティック回帰の学習も予測も軽く速くなります——顔認識システムが固有顔で実用的に動く理由が、この一手です。

**別の見方：圧縮とノイズ除去は同じ操作**　小さい固有値の主成分を捨てることは、「情報の少ない向き」を落とすと同時に、そこに乗っている**細かなノイズを一緒に落とす**ことでもあります。だから PCA は、次元削減の道具であると同時に**ノイズ除去**の道具でもあります。圧縮・可視化・ノイズ除去は別々のご利益に見えて、「分散の小さい向きを捨てる」という一つの操作の別の顔なのです。ただし顔の細部（本物の個性）と雑音の境目は曖昧で、削りすぎれば個性まで落ちる——前節の累積寄与率と再構成画像で、どこまで削ってよいかを必ず目で確かめる姿勢が要ります。

**つまずきどころ：主成分の符号は決められない**　固有値方程式 $\Sigma v = \lambda v$ は、$v$ を符号反転した $-v$ でも同じように成り立ちます（$\Sigma(-v)=\lambda(-v)$）。つまり**主成分の向きの符号は本質的に任意**で、ライブラリは内部の約束で片方に決めているだけ。データや実行環境が変われば、同じ固有顔が明暗反転して出ることもあります。ただし心配は要りません。ある固有顔 $v_k$ の符号を反転しても、対応するスコア $s_k = \tilde{x}\cdot v_k$ も同時に符号が変わるため、積 $s_k v_k$ は不変で、**再構成される顔はまったく同じ**です（実際に符号を反転して再構成しても画像はぴたり一致します）。注意すべきは解釈のときで、固有顔の「こちらが明るく、あちらが暗い」を**絶対的な意味**として読まないこと。意味があるのは向き（軸）であって、その符号ではありません。前節の「固有顔を1枚だけで意味づけすぎない」と合わせて、PCA の解釈はいつも控えめに、が鉄則です。

**発展：白色化（whitening）で軸のスケールをそろえる**　主成分に射影しただけでは、各スコア軸の分散は固有値 $\lambda_k$ のまま**バラバラ**です（このデータでは第1〜5成分でおよそ 18.8, 11.1, 6.3, 4.0, 2.9）。第1成分は第5成分の6倍以上の分散を持つわけで、距離や内積で軸を対等に扱いたい後段の手法にそのまま渡すと、大きい軸ばかりが効いてしまいます。そこで各軸を $\sqrt{\lambda_k}$ で割り、**すべての軸の分散を1にそろえる**のが白色化です（`PCA(whiten=True)`）。そろえた後の各軸の分散はきれいに 1 になります。次回以降のクラスタリングのように距離を測る手法へ渡す前処理として有効ですが、小さい固有値の軸まで引き伸ばすとノイズも増幅されるので、使う成分数とセットで考える控えめな道具です。

**忘れずに：PCA が捉えるのは「まっすぐな方向」だけ**　ここまで見た強みは、すべて $\Sigma v=\lambda v$ という**線形**の枠組みから来ています。裏を返せば、PCA が見つけるのは直線的な（平らな）方向だけ。データが渦巻きやドーナツのように曲がって広がっているとき、その構造を1本の直線軸では表せません。顔画像がこれだけうまく圧縮できたのは、顔の違いの多くが「明るさ」「向き」といった**おおむね直線的に効く成分**で説明できたからでもあります。曲がった構造を相手にするときは、非線形の次元削減が別に必要になる——PCA は万能ではなく、まっすぐさを仮定した強力な道具だ、と位置づけておきましょう。


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score
full = cross_val_score(LogisticRegression(max_iter=2000), X, y, cv=5).mean()
red = cross_val_score(Pipeline([('pca', PCA(n_components=50, random_state=0)),
                                ('m', LogisticRegression(max_iter=2000))]), X, y, cv=5).mean()
print(f'4096次元すべて: 正解率 {full:.3f}')
print(f'PCAで50次元  : 正解率 {red:.3f}')

## 目標に答えられたか

- 今日の目標は「ラベルなしで、4096次元の顔を少数の軸で捉え、圧縮したい」でした
- 1節の固有顔は、どんな「顔の部品」を表していましたか？（明るさ・向き・メガネなど）
- TODO①で、上位50個の主成分は全体の何割の情報を保てましたか？
- TODO②で、成分数を増やすと再構成はどう変わりましたか？ 何成分あれば個人が分かりましたか？
- 3節で、4096次元を50次元に減らしても正解率は保てましたか？ それは何を意味するでしょう？

## 課題（提出）

**提出するもの**: 応用②の答えと、応用③の文章。提出フォームに入力してください。期限はありません。応用①のコードは提出しませんが、②の答えを出すために必要です。


### 応用①（変形）

3節では PCA の成分数を 50 に決め打ちして人物当てをしました。今度は成分数を **10, 50, 100** の3通りに変えます。
3節と同じ手順（`PCA` → `LogisticRegression` の `Pipeline` を5分割交差検証）で、それぞれの正解率を小数第3位まで表示してください。


In [ ]:
accs = {}
for k in [10, 50, 100]:
    accs[k] = cross_val_score(Pipeline([('pca', PCA(n_components=k, random_state=0)),
                                        ('m', LogisticRegression(max_iter=2000))]), X, y, cv=5).mean()
    print(f'PCAで{k:3d}次元: 正解率 {accs[k]:.3f}')


<details><summary>詰まったら</summary>

3節で `red` を求めた行の `n_components=50` を変えるだけです。`for k in [10, 50, 100]:` で回すと1回で書けます。正解率は `{k: 値}` の辞書に入れておくと応用②で使えます。

</details>


### 応用②（判断）

応用①の3つのうち、正解率が**初めて 0.95 を超える成分数**はどれですか。10・50・100 の**整数**で答えてください。


In [ ]:
best = min(k for k in accs if accs[k] > 0.95)   # 0.95 を超える成分数のうち最小
print('答え:', best)


<details><summary>詰まったら</summary>

表示した3つの値を小さい順に見て、最初に 0.95 を超えた成分数を答えます。コードで選ぶなら、辞書 `accs` に対して `min(k for k in accs if accs[k] > 0.95)` で、0.95 を超える成分数のうち最小のものが取れます。

</details>


### 応用③（解釈）

成分数を増やせば、人物当ての正解率は必ず上がるのでしょうか。応用①の3つの正解率と、TODO①で求めた累積寄与率（`np.cumsum(pca.explained_variance_ratio_)` の配列から 10・50・100 番目を読みます）の2つを根拠に、顔認証システムを作る開発チームに向けて3行で書いてください。


**模範例**

正解率は 10成分で 0.872、50成分で 0.972、100成分で 0.970 でした。10 から 50 へは大きく上がりますが、50 から 100 へはほぼ横ばいです（この実行では 0.972 → 0.970）。

累積寄与率は 10成分で約66%、50成分で約87%、100成分で約94% です。成分を増やせば保てる情報は増えますが、後ろの成分が担うのは細部やノイズで、人物の見分けにはほとんど効きません。

「多いほど良い」わけではないので、成分数は正解率が頭打ちになる 50 前後で止め、計算の軽さを取ることを勧めます。


## 発展（任意）

### UMAP で400枚の顔を2次元に落とす

3節の深掘りの最後で、PCA が捉えるのは「まっすぐな方向」だけだと述べました。実際、PCA の上位2成分だけで散布図にすると、明るさや向きの違いで点は広がりますが、同じ人物の10枚は中央でほかの人物と混ざってしまいます。

**UMAP** は、高次元で近くにあった点どうしを低次元でも近くに置くことを目標にした、非線形の次元削減です。直線の軸を探すのではなく、「近い点のつながり」を保ちながら平面に敷き詰めるので、同じ人物のようなまとまりが見えやすくなります。

ただし PCA と違い、軸に「分散が最大の方向」のような意味はなく、塊どうしの離れ具合も、そのまま元の距離とは読めません。可視化の道具として割り切って使います。PCA 2成分と UMAP の散布図を、人物 ID で色分けして並べます。


In [ ]:
# Colab に入っていないことがあるので、なければインストールする
import importlib.util
if importlib.util.find_spec('umap') is None:
    %pip install -q umap-learn


In [ ]:
import warnings
import umap   # パッケージ名は umap-learn、import 名は umap

warnings.filterwarnings('ignore', message='n_jobs value')   # random_state を固定したときの注意を抑える
emb_pca = pca.transform(X)[:, :2]   # 1節で学習済みの pca の第1・第2主成分スコア
emb_umap = umap.UMAP(n_components=2, random_state=42).fit_transform(X)   # 初回は数十秒かかる

fig, axes = plt.subplots(1, 2, figsize=(11, 4.6))
for ax, Z, title in zip(axes, [emb_pca, emb_umap], ['PCA（2成分）', 'UMAP（2次元）']):
    ax.scatter(Z[:, 0], Z[:, 1], c=y, cmap='tab20', s=14)   # 色 = 人物 ID
    ax.set_title(title)
plt.tight_layout(); plt.show()


In [ ]:
# 「同じ人物がまとまっているか」を数にする：各点の最近傍が同じ人物である割合
from sklearn.neighbors import NearestNeighbors
for name, Z in [('PCA 2成分', emb_pca), ('UMAP 2次元', emb_umap)]:
    nn = NearestNeighbors(n_neighbors=2).fit(Z)
    nearest = nn.kneighbors(Z, return_distance=False)[:, 1]   # 0列目は自分自身
    print(f'{name}: 最近傍が同じ人物である割合 {np.mean(y[nearest] == y):.3f}')


**読み方**　左の PCA では、点は横に広がるものの、同じ色（同じ人物）の点は中央で入り混じります。右の UMAP では、同じ色の点が小さな塊になって平面に散らばり、人物ごとのまとまりが目で分かります。

数で見ると、最近傍が同じ人物である割合は PCA 2成分で 0.297、UMAP で 0.805 です。同じ400枚を2次元に落としても、「近い点を近くに保つ」目標の違いで、これだけ差が出ます。

注意点が3つあります。色は20色を40人で使い回しているので、同じ色でも別人のことがあります。UMAP の軸の目盛りには意味がなく、塊どうしの離れ具合も、そのまま元の距離とは読めません。`random_state` を変えると図の形も変わります（まとまり方の傾向は変わりません）。

3節の人物当てで使ったのは PCA の50次元でした。UMAP は「見る」ための道具、PCA は「圧縮して次の手法に渡す」道具、と使い分けるのが、まずの目安です。

試すなら、`umap.UMAP(n_neighbors=5, ...)` や `n_neighbors=50` のように近傍の数（既定は 15）を変えて、塊の大きさや配置がどう変わるか見てください。
